# EDA Recommendation trên HDFS — từng category

Đặc tả và hướng dẫn chuẩn: [docs/eda.md](../docs/eda.md). Giữ nguyên `01_eda.ipynb` để tham khảo; các thống kê 25.000 dòng đầu ở đó không đại diện toàn bộ dataset. EDA chỉ đọc raw, chưa tạo processed/features/model. Count và distinct tính toàn phạm vi; quantile xấp xỉ accuracy=10000.

In [ ]:
from pathlib import Path
import sys

module_dir = Path.cwd() if (Path.cwd() / 'eda_recommendation.py').exists() else Path.cwd() / 'notebooks'
sys.path.insert(0, str(module_dir))
from eda_recommendation import get_scope, run_eda
scope = get_scope()

SELECTED_CATEGORIES = scope['categories'][:1]  # Mỗi lượt chỉ một category trong scope.
BASE = 'hdfs://namenode:8020' + scope['hdfs_base']
VERSION = scope['version']
OUTPUT = '/workspace/outputs/eda'
SPLIT_DATE = '2022-01-01'  # Diagnostic, chưa chốt split của model.
RUN_FIVE_CORE = False
CORES = 1
SHUFFLE_PARTITIONS = 64
MIN_FREE_GIB = 20
# Cell cấu hình không khởi tạo Spark.

## Chạy EDA một category

CLI và notebook dùng cùng run_eda: local[1], driver 1 GiB, shuffle 64, Spark local dir trong /tmp. Mở Jupyter với compose.eda-light.yaml để có container RAM 2 GiB/CPU 2. Chỉ chạy một kernel/job; nếu có Spark session cũ thì dừng hoặc restart kernel trước.

Mỗi run có run.json và resources.json; kiểm tra dung lượng ở ranh giới các pha. Guard không phải quota và không bảo đảm một Spark action luôn vừa RAM/đĩa. Batch PowerShell còn giám sát ổ host chứa outputs/VHDX mỗi 5 giây. Không có cross-category scratch hoặc global overlap trong đường chạy này.

In [ ]:
report_dir = run_eda(categories=SELECTED_CATEGORIES, base=BASE, version=VERSION, output=OUTPUT,
                     split_date=SPLIT_DATE, run_core=RUN_FIVE_CORE, min_free_gb=MIN_FREE_GIB,
                     cores=CORES, shuffle_partitions=SHUFFLE_PARTITIONS)
# run_eda dừng Spark trong finally, kể cả khi lỗi hoặc ngắt.
print('Báo cáo:', report_dir)

## Đọc insight cho processing

1. `*_quality`, `*_missing`: absent/null/empty/placeholder/native type/convertibility/domain/nested shape và trace file + SHA256 dòng.
2. `duplicates`, `metadata_conflict_classes`: copies, event conflicts, repeated pairs; complementary chưa tự merge.
3. `rating_*`, `price_*`: phân bố rating hợp lệ; price cấp singleton/exact-copy item, unresolved keys báo riêng.
4. `activity_*`, `timestamp_*`, `user_history`, `interaction_gaps`: UTC, seconds/milliseconds, gaps=0, tháng rỗng và tháng biên.
5. `sparsity`, `*_degree_*`, `long_tail*`, `user_threshold_impact`: distinct neighbors, density và retention sau ngưỡng.
6. `metadata_coverage`, `coverage_*`, `matched_content`: membership khác safe match; assert số dòng trước/sau join bằng nhau.
7. `processing_gate_impact`, `temporal_*`: tác động độc lập/cộng dồn; warm/cold user/item và pair đã thấy trước cutoff.
8. `comparison`: summary và sparsity của category đang chạy. Overlap/global distinct chưa đo; không cộng số user từng category thành unique users toàn scope.

Metadata là snapshot crawling: price/average_rating/rating_number có thể gây leakage. Review không phải thời điểm mua hàng; first_seen không phải ngày tạo tài khoản/ra mắt. Đọc retention và cold-start trước khi chọn min-history; chưa tự quyết định policy xử lý.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Markdown, display, Image

REPORT_DIRECTORY = None  # Đặt '/workspace/outputs/eda/run_...' để xem report cũ độc lập.
selected_report = Path(REPORT_DIRECTORY) if REPORT_DIRECTORY else globals().get('report_dir')
if selected_report is None:
    raise ValueError('Điền REPORT_DIRECTORY hoặc chạy một category trước.')
info = json.loads((selected_report / 'run.json').read_text())
print(info)
# Xem report không import/khởi tạo Spark; failed run chỉ hiện category đã hoàn tất.
for category in info['completed_categories']:
    directory = selected_report / category
    display(Markdown(f'### {category}'))
    display(pd.read_csv(directory / 'processing_actions.csv'))
    display(pd.read_csv(directory / 'processing_gate_impact.csv'))
    display(Image(filename=str(directory / 'reviews_missing.png')))
    display(Image(filename=str(directory / 'user_degree_bins.png')))